# Nemotron 3 Nano — SFT Warm-Start (RTX 6000 Pro v13)

Single-GPU SFT on `data/merged_cot_final.csv`. Goal: imprint chat template +
`<think>...</think>\boxed{}` format on the base Nemotron-3-Nano-30B model.

**Pair**: GRPO done-right runs after this in
`post-training/nemo-v13-drgrpo-rtx6000.ipynb` (loads the adapter saved here).

**Hardware**: 1× RTX 6000 Pro (Blackwell, ~96 GB VRAM). Bare HuggingFace —
no Unsloth (Blackwell PTXAS path is fragile).

## Pipeline position

```
THIS NOTEBOOK ─── SFT warm-start ──> SFT_ADAPTER_DIR
                                      │
                                      ▼
                                post-training/nemo-v13-drgrpo-rtx6000.ipynb
                                  (Dr.GRPO + verifiable rewards)
                                      │
                                      ▼
                                GRPO_ADAPTER_DIR ──> submission.zip
```


## Path & Run Configuration


In [ ]:
import os, sys

os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# Repo layout — override via env var if running elsewhere
REPO_ROOT = os.environ.get("KAGGLE_NEMO_REPO", r"F:/Hackathons/Kaggle-Nemotron")

BASE_MODEL_NAME   = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
SFT_DATA_PATH     = os.path.join(REPO_ROOT, "data", "merged_cot_final.csv")
GRPO_DATA_PATH    = os.path.join(REPO_ROOT, "data", "src", "train.csv")

OUTPUT_ROOT       = os.path.join(REPO_ROOT, "outputs", "rtx6000_v13")
SFT_ADAPTER_DIR   = os.path.join(OUTPUT_ROOT, "sft_adapter")
GRPO_ADAPTER_DIR  = os.path.join(OUTPUT_ROOT, "grpo_adapter")
SUBMISSION_DIR    = os.path.join(OUTPUT_ROOT, "submission_adapter_sft")
TB_LOG_DIR        = os.path.join(OUTPUT_ROOT, "tb_logs_sft")

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(TB_LOG_DIR, exist_ok=True)

SEED          = 42
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

print({
    "REPO_ROOT": REPO_ROOT,
    "BASE_MODEL_NAME": BASE_MODEL_NAME,
    "SFT_DATA_PATH": SFT_DATA_PATH,
    "GRPO_DATA_PATH": GRPO_DATA_PATH,
    "SFT_ADAPTER_DIR": SFT_ADAPTER_DIR,
    "GRPO_ADAPTER_DIR": GRPO_ADAPTER_DIR,
})


## One-time dependency install
Run once with `INSTALL_DEPS=True`, restart kernel, set back to `False`.


In [ ]:
# Run ONCE per environment, then restart kernel and set back to False.
INSTALL_DEPS = False

if INSTALL_DEPS:
    import subprocess, sys
    pkgs = [
        # Blackwell sm_120/sm_100: need recent torch with CUDA 12.4+.
        # If wheels missing for your Blackwell SKU, swap to nightly:
        #   pip install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu124
        "torch>=2.4",
        "transformers>=4.46",
        "peft>=0.13",
        "trl>=0.16",            # required for loss_type='dr_grpo' / scale_rewards / mask_truncated
        "datasets>=3.0",
        "accelerate>=1.0",
        "bitsandbytes>=0.44",
        "tensorboard",
        "mamba-ssm",
        "causal-conv1d",
        "pandas", "numpy",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-U"] + pkgs, check=True)
    print("Install done. RESTART the kernel before continuing.")
else:
    print("INSTALL_DEPS=False — assuming env is ready.")


## Model + Tokenizer Loading


In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required (target: RTX 6000 Pro Blackwell).")

dev_props = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name()}  Total VRAM: {dev_props.total_memory/1e9:.1f} GB  CC: sm_{dev_props.major}{dev_props.minor}")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model (bf16, eager attention)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
    attn_implementation="eager",
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

total_b = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded — {total_b:.2f}B params total.  Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## LoRA Target Discovery


In [ ]:
import re
from collections import Counter

linear_modules = []
for name, mod in model.named_modules():
    cls = mod.__class__.__name__
    if cls in ("Linear", "Linear4bit", "Linear8bitLt"):
        linear_modules.append(name)

suffix_counts = Counter(n.rsplit(".", 1)[-1] for n in linear_modules)
print("Linear suffix counts (top 20):")
for s, c in suffix_counts.most_common(20):
    print(f"  {s:30s} {c}")

parent_counts = Counter()
for n in linear_modules:
    parts = n.split(".")
    if len(parts) >= 2:
        parent_counts[parts[-2]] += 1
print("\nParent module counts (top 20):")
for p, c in parent_counts.most_common(20):
    print(f"  {p:30s} {c}")

print("\nSample names:")
for n in linear_modules[:3] + linear_modules[len(linear_modules)//2:len(linear_modules)//2+3] + linear_modules[-3:]:
    print(f"  {n}")


## LoRA Config (rank=32, RSLoRA)


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

LORA_RANK    = 32
LORA_ALPHA   = 64
LORA_DROPOUT = 0.0

# Sensitive targets only — excludes routable experts (sparse), router (frozen by NVIDIA),
# lm_head / embeddings (destabilize). See CLAUDE.md LoRA priority table.
target_regex = (
    r".*("
    r"self_attn\.(q|k|v|o)_proj"
    r"|mamba\.(in|out|x|dt)_proj"
    r"|shared_expert\.(gate|up|down)_proj"
    r")$"
)

matched = [n for n in linear_modules if re.match(target_regex, n)]
print(f"Regex matched {len(matched)} modules.")
if not matched:
    print("[WARN] regex matched 0 modules — module names differ from expected.")
    print("       Falling back to suffix list (will also hit routable experts — wasteful).")
    target_modules = ["q_proj","k_proj","v_proj","o_proj","in_proj","out_proj","x_proj","dt_proj","gate_proj","up_proj","down_proj"]
else:
    print("Sample matches:", matched[:3], "...", matched[-2:])
    target_modules = target_regex

lora_config = LoraConfig(
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    target_modules = target_modules,
    task_type      = TaskType.CAUSAL_LM,
    use_rslora     = True,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## SFT Dataset Prep — `merged_cot_final.csv`

Builds conversational records `[user, assistant]` where the assistant turn is
the canonical `<think>{cot}</think>\boxed{{answer}}` template. Strips any
pre-existing `\boxed{{}}` from the CoT before re-appending a clean one.


In [ ]:
import pandas as pd, re
from datasets import Dataset as HFDataset

df_sft = pd.read_csv(SFT_DATA_PATH)
print(f"SFT data: {len(df_sft)} rows.  Columns: {list(df_sft.columns)}")
required = {"prompt", "cot", "answer"}
missing = required - set(df_sft.columns)
if missing:
    raise ValueError(f"Missing required columns in {SFT_DATA_PATH}: {missing}")

df_sft = df_sft.dropna(subset=list(required)).reset_index(drop=True)
df_sft = df_sft.sample(frac=1, random_state=SEED).reset_index(drop=True)

_BOXED_STRIP = re.compile(r"\\boxed\{[^}]*\}")
def clean_cot(c):
    c = _BOXED_STRIP.sub("", str(c)).rstrip()
    return c.replace("<think>", "").replace("</think>", "").strip()

records, skipped = [], 0
for _, row in df_sft.iterrows():
    cot = clean_cot(row["cot"])
    if len(cot) < 5:
        skipped += 1
        continue
    user_content = str(row["prompt"]) + PROMPT_SUFFIX
    assistant_content = f"<think>\n{cot}\n</think>\n\\boxed{{{row['answer']}}}"
    records.append({"messages": [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]})

sft_dataset = HFDataset.from_list(records)
print(f"SFT records: {len(records)} (skipped {skipped} short / empty CoT)")


## SFT Trainer

Config matches the v7-5 Kaggle settings but scaled for RTX 6000 Pro headroom:

| Param | Value | Note |
|---|---|---|
| `num_train_epochs` | 2 | enough to imprint format |
| `per_device_train_batch_size` | 2 | 96 GB VRAM allows |
| `gradient_accumulation_steps` | 4 | effective batch = 8 |
| `learning_rate` | 5e-5 | matches NVIDIA's SFT recipe |
| `lr_scheduler_type` | cosine | with 5% warmup |
| `max_length` | 4096 | bigger than v7-5's 2048; covers long CoT |
| `optim` | paged_adamw_8bit | memory-friendly |
| `packing` | True | densify short samples |


In [ ]:
# ============================================================
# Mirrors working SFT config from sft/nvidia-nemotron-v7-5.ipynb
# Key fixes vs prior v13:
#   - packing=False         (packing+GC+paged_adamw caused step ~1330 crash)
#   - use_reentrant=True    (Mamba + non-reentrant GC is fragile)
#   - PYTORCH_ALLOC_CONF=expandable_segments:True
#   - TORCH_CUDA_ALLOC_CONF=max_split_size_mb:128
#   - dataloader_num_workers=2, report_to=none, max_length=8192
#   - save_steps checkpoints so crash != lost run
# ============================================================
import os
os.environ["PYTORCH_ALLOC_CONF"]    = "expandable_segments:True"
os.environ["TORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import gc, time, torch
from trl import SFTTrainer, SFTConfig

MAX_SEQ_LEN_SFT = 8192

def sft_formatting(example):
    msgs = example["messages"]
    if msgs and isinstance(msgs[0], dict):
        convs = [msgs]
    else:
        convs = msgs
    out = []
    for c in convs:
        try:
            t = tokenizer.apply_chat_template(c, tokenize=False,
                                              add_generation_prompt=False,
                                              enable_thinking=True)
        except TypeError:
            t = tokenizer.apply_chat_template(c, tokenize=False,
                                              add_generation_prompt=False)
        out.append(t)
    return out

sft_args = SFTConfig(
    output_dir                   = os.path.join(OUTPUT_ROOT, "sft_run"),
    num_train_epochs             = 2,
    per_device_train_batch_size  = 2,
    gradient_accumulation_steps  = 4,         # effective batch = 8
    learning_rate                = 8e-5,      # v7-5 value
    lr_scheduler_type            = "cosine",
    warmup_ratio                 = 0.05,
    max_length                   = MAX_SEQ_LEN_SFT,
    packing                      = False,     # FIX: was True -> crash
    optim                        = "paged_adamw_8bit",
    adam_beta1                   = 0.9,
    adam_beta2                   = 0.95,
    adam_epsilon                 = 1e-8,
    weight_decay                 = 0.01,
    max_grad_norm                = 1.0,
    bf16                         = True,
    gradient_checkpointing       = True,
    gradient_checkpointing_kwargs= {"use_reentrant": True},   # FIX: was False
    logging_steps                = 10,
    save_strategy                = "steps",
    save_steps                   = 200,
    save_total_limit             = 3,
    dataloader_num_workers       = 2,
    remove_unused_columns        = False,
    seed                         = SEED,
    report_to                    = "none",    # tensorboard interacts badly w/ unsloth on kaggle
    dataset_num_proc             = 4,
)

sft_trainer = SFTTrainer(
    model            = model,
    args             = sft_args,
    train_dataset    = sft_dataset,
    processing_class = tokenizer,
    formatting_func  = sft_formatting,
)

torch.cuda.empty_cache(); gc.collect()

# Auto-resume if any checkpoint already on disk
_run_dir = sft_args.output_dir
_ckpts = []
if os.path.isdir(_run_dir):
    _ckpts = [d for d in os.listdir(_run_dir) if d.startswith("checkpoint-")]
resume = bool(_ckpts)
print(f"Resuming from checkpoint: {resume}  ({sorted(_ckpts)[-3:] if _ckpts else 'none'})")

print("Starting SFT warm-start...")
t0 = time.time()
sft_trainer.train(resume_from_checkpoint=resume)
print(f"SFT done in {(time.time()-t0)/60:.1f} min")

os.makedirs(SFT_ADAPTER_DIR, exist_ok=True)
model.save_pretrained(SFT_ADAPTER_DIR)
tokenizer.save_pretrained(SFT_ADAPTER_DIR)
print(f"SFT adapter saved -> {SFT_ADAPTER_DIR}")


## Post-SFT Sanity Check (3-sample greedy generation)

Before launching multi-hour GRPO, verify the SFT adapter actually emits
`\boxed{}`. If this fails → fix data/formatter before running GRPO.


In [ ]:
import torch
model.eval()
sample_rows = df_sft.sample(3, random_state=0)[["prompt", "answer"]].values.tolist()
for p, ans in sample_rows:
    msgs = [{"role": "user", "content": str(p) + PROMPT_SUFFIX}]
    try:
        text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                              add_generation_prompt=True,
                                              enable_thinking=True)
    except TypeError:
        text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                              add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    has_box = "\\boxed{" in gen
    tag = "BOX  " if has_box else "NOBOX"
    print(f"[{tag}] expected={str(ans)!r:30s}  tail={gen[-180:]!r}")
model.train()


## (Optional) Package SFT-only submission

Useful for an early leaderboard read on what SFT alone achieves before GRPO.
Skip if you intend to run GRPO immediately.


In [ ]:
PACKAGE_SFT_SUBMISSION = False  # flip to True if you want a pre-GRPO submission

if PACKAGE_SFT_SUBMISSION:
    import json, shutil, zipfile
    os.makedirs(SUBMISSION_DIR, exist_ok=True)
    required = ["adapter_config.json", "adapter_model.safetensors"]
    for fname in required:
        sp = os.path.join(SFT_ADAPTER_DIR, fname)
        dp = os.path.join(SUBMISSION_DIR, fname)
        if not os.path.exists(sp):
            raise FileNotFoundError(f"Missing: {sp}")
        shutil.copy2(sp, dp)
        print(f"  copied {fname}  ({os.path.getsize(dp)/1024/1024:.1f} MB)")
    cfg_path = os.path.join(SUBMISSION_DIR, "adapter_config.json")
    with open(cfg_path) as f: cfg = json.load(f)
    cfg["base_model_name_or_path"] = BASE_MODEL_NAME
    cfg["inference_mode"] = True
    cfg["lora_dropout"]   = 0.0
    with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)
    zip_path = os.path.join(OUTPUT_ROOT, "submission_sft.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in required:
            zf.write(os.path.join(SUBMISSION_DIR, fname), fname)
    print(f"\nsubmission_sft.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB")
else:
    print("PACKAGE_SFT_SUBMISSION=False — skipping. Run GRPO notebook next.")


## Next step

Open `post-training/nemo-v13-drgrpo-rtx6000.ipynb` and run it. It will load
the SFT adapter from `SFT_ADAPTER_DIR`, apply Dr. GRPO with verifiable
per-puzzle-type rewards, and produce the final `submission.zip`.
